# EDA v3 — Evolution 2018 → 2025 (Modele CLV Nouvelles Acquisitions)

Notebook **pas-a-pas** et **lisible** : une cellule construit un **DataFrame explicite**
(a inspecter), la cellule suivante trace **un seul graphe** interactif (Plotly).
Angle : comment les choses **evoluent d'annee en annee** (acquisitions, NBI, encours, marge, churn).

> Graphes exportes en **HTML** dans `../eda_html_v3/` (zoom + survol → partageables sans le notebook).
> `pip install plotly` requis.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"
CLEAN_DIR = Path("../clean_data")
HTML_DIR  = Path("../eda_html_v3"); HTML_DIR.mkdir(exist_ok=True)

EPOCH_YEAR = 1994
def idx_to_year(idx): return EPOCH_YEAR + (int(idx) - 1) // 12
NBI_COL = "revenus_eur_avec_liquidity"

FAM_ORDER  = ["epargne", "invest", "credit_immo", "autres_cr", "cartes"]
ENC_FAMS   = ["epargne", "invest", "credit_immo", "autres_cr"]
FAM_LABELS = {"epargne": "Epargne", "invest": "Investissements", "credit_immo": "Credit immo",
              "autres_cr": "Autres credits", "cartes": "Autre PNB"}
FAM_COLOR  = {"Epargne": "#06B6D4", "Investissements": "#7C3AED", "Credit immo": "#EC4899",
              "Autres credits": "#F59E0B", "Autre PNB": "#84CC16"}
FAMILY_ENCOURS = {
    "epargne":      ("savings",   ["avoirs_eur"]),
    "invest":       ("invest",    ["cp_depots_a_terme", "cp_depots_en_cc_epar_liq",
                                   "cp_depots_en_cc_placement", "cp_depots_titres", "cp_epargne_a_vue"]),
    "invest_banca": ("banca",     ["capitaux_productifs_eur"]),
    "credit_immo":  ("pret_immo", ["credits_utilises_eur"]),
    "autres_cr":    ("credit",    ["credits_utilises_eur"]),
}

CARACTS = ["canal", "tranche_revenu", "residence", "tranche_age", "csp"]   # caract. x pour les vues combinees

def save_html(fig, name):
    """Sauve le graphe en HTML (partageable) et l'affiche dans le notebook."""
    fig.write_html(HTML_DIR / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

def strip_facet_titles(fig):
    """'caracteristique=canal' -> 'canal' sur les titres de facettes."""
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    return fig

print("Setup OK — HTML ->", HTML_DIR.resolve())

## 0. Fondations — chargement & objets de base

On charge les fichiers propres, on construit le **panel famille** (encours + NBI par famille
et par mois), l'axe **tenure** (mois depuis l'acquisition), la table **client** `cli`
(1 ligne/client avec ses caracteristiques), et les objets de **churn corriges** (censure par
cohorte — pas de saut a 100%).

In [ ]:
def load(name):
    p = CLEAN_DIR / f"clean_{name}.parquet"
    if not p.exists():
        print(f"  MANQUANT: {p.name}"); return None
    df = pd.read_parquet(p); print(f"  {name:<12}: {len(df):>10,} lignes"); return df

print("Chargement:")
stock  = load("stock"); flux = load("flux")
savings = load("savings"); invest = load("invest"); banca = load("banca")
cartes = load("cartes"); credit = load("credit"); pret_immo = load("pret_immo")
upstream = load("upstream")
HAS_UPSTREAM = upstream is not None and len(upstream) > 0

In [ ]:
# Acquisition, tenure, alias NBI, annee calendaire
acq = stock.groupby("identifiant_tiers")["periode"].min().rename("acq_period")
def add_tenure(df):
    if df is None: return None
    df = df.merge(acq, on="identifiant_tiers", how="left")
    df["tenure"] = (df["periode"] - df["acq_period"]).astype("Int64")
    return df
stock = add_tenure(stock); flux = add_tenure(flux)
PRODUCTS = {"savings": savings, "invest": invest, "banca": banca, "cartes": cartes,
            "credit": credit, "pret_immo": pret_immo}
for k in list(PRODUCTS): PRODUCTS[k] = add_tenure(PRODUCTS[k])
savings, invest, banca, cartes, credit, pret_immo = (PRODUCTS[k] for k in
    ["savings", "invest", "banca", "cartes", "credit", "pret_immo"])
if NBI_COL in flux.columns: flux["nbi"] = flux[NBI_COL]
cohort_of = acq.map(idx_to_year)
print("Cohortes:", sorted(cohort_of.dropna().unique()))

In [ ]:
# Panel famille : (client, periode, famille) -> encours + nbi
def agg_encours(df, cols):
    cols = [c for c in cols if df is not None and c in df.columns]
    if not cols: return None
    t = df[["identifiant_tiers", "periode"]].copy(); t["encours"] = df[cols].sum(axis=1, min_count=1)
    return t.groupby(["identifiant_tiers", "periode"], as_index=False)["encours"].sum()
def agg_nbi(df):
    if df is None or NBI_COL not in df.columns: return None
    t = df[["identifiant_tiers", "periode", NBI_COL]].rename(columns={NBI_COL: "nbi"})
    return t.groupby(["identifiant_tiers", "periode"], as_index=False)["nbi"].sum()
def combine(a, b, v):
    parts = [x for x in (a, b) if x is not None]
    return pd.concat(parts).groupby(["identifiant_tiers", "periode"], as_index=False)[v].sum() if parts else None

enc = {"epargne": agg_encours(savings, FAMILY_ENCOURS["epargne"][1]),
       "invest": combine(agg_encours(invest, FAMILY_ENCOURS["invest"][1]),
                        agg_encours(banca, FAMILY_ENCOURS["invest_banca"][1]), "encours"),
       "credit_immo": agg_encours(pret_immo, FAMILY_ENCOURS["credit_immo"][1]),
       "autres_cr": agg_encours(credit, FAMILY_ENCOURS["autres_cr"][1])}
nbi = {"epargne": agg_nbi(savings), "invest": combine(agg_nbi(invest), agg_nbi(banca), "nbi"),
       "credit_immo": agg_nbi(pret_immo), "autres_cr": agg_nbi(credit), "cartes": agg_nbi(cartes)}
parts = []
for fam in FAM_ORDER:
    n, e = nbi.get(fam), enc.get(fam)
    if n is None and e is None: continue
    base = n.copy() if n is not None else e[["identifiant_tiers", "periode"]].copy()
    if n is None: base["nbi"] = np.nan
    base = base.merge(e, on=["identifiant_tiers", "periode"], how="outer") if e is not None else base.assign(encours=np.nan)
    base["family"] = fam; parts.append(base)
panel = pd.concat(parts, ignore_index=True).merge(acq, on="identifiant_tiers", how="left").dropna(subset=["acq_period"])
panel["tenure"] = (panel["periode"] - panel["acq_period"]).astype(int)
panel["annee"]  = panel["periode"].map(idx_to_year)          # annee calendaire
panel["cohorte"] = panel["acq_period"].map(idx_to_year)      # annee d'acquisition
panel = panel[panel["tenure"] >= 0]
panel["famille"] = panel["family"].map(FAM_LABELS)
early = panel[panel["tenure"].between(0, 2)]                  # 3 premiers mois (M+0, M+1, M+2)
print(f"Panel: {len(panel):,} lignes | annees {panel['annee'].min()}-{panel['annee'].max()}")

In [ ]:
# Table client `cli` : 1 ligne par client, caracteristiques au dernier snapshot
latest = stock.sort_values("periode").groupby("identifiant_tiers").last()
REV_ORDER = ["Moins de 35.999 EUR", "Entre 36.000 et 53.999 EUR",
             "Entre 54.000 et 119.999 EUR", "Plus de 120.000 EUR"]
AGE_LABELS = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

cli = pd.DataFrame(index=latest.index)
cli["cohorte"] = cohort_of.reindex(cli.index)
cli["canal"]   = latest["canal"] if "canal" in latest.columns else "Inconnu"
rev = latest["revenu_total"].astype(str).str.strip() if "revenu_total" in latest.columns else pd.Series("Inconnu", index=cli.index)
cli["tranche_revenu"] = pd.Categorical(rev.where(rev.isin(REV_ORDER), "Inconnu"),
                                        categories=REV_ORDER + ["Inconnu"], ordered=True)
cli["residence"] = (latest["frontalier_o_n"].astype(str).str.strip().str.upper()
                    .map({"O": "Frontalier", "N": "Resident"}).fillna("Inconnu")
                    if "frontalier_o_n" in latest.columns else "Inconnu")
age = pd.to_numeric(latest["age"], errors="coerce") if "age" in latest.columns else pd.Series(np.nan, index=cli.index)
cli["age_num"] = age
cli["tranche_age"] = pd.cut(age, [-1, 17, 24, 34, 44, 54, 64, 200], labels=AGE_LABELS)
cli["csp"] = latest["segment_ameliore"].astype(str).str.strip() if "segment_ameliore" in latest.columns else "Inconnu"

EXCLUDE_MINORS = True     # age < 18 exclus (hors mineurs)
if EXCLUDE_MINORS:
    cli = cli[(cli["age_num"] >= 18) | cli["age_num"].isna()]

# ── Ordre des modalites (EDITABLE) : change une liste pour re-ordonner un axe ──
ORDERS = {
    "canal":          ["Digital", "Physique", "Inconnu"],
    "tranche_revenu": REV_ORDER + ["Inconnu"],
    "residence":      ["Frontalier", "Resident", "Inconnu"],
    "tranche_age":    AGE_LABELS,
    "csp":            list(cli["csp"].value_counts().index),   # par frequence — remplace par ta liste pour imposer un ordre
}
# Ordre unique pour les axes 'categorie' des vues combinees (facettes)
UNION_ORDER = (ORDERS["tranche_revenu"] + ORDERS["tranche_age"] + ORDERS["canal"]
               + ORDERS["residence"] + ORDERS["csp"])
print(f"Clients dans cli : {len(cli):,}")
print("ORDERS pret — edite ORDERS['csp'] (ou autre) pour imposer un ordre manuel.")

In [ ]:
# Churn CORRIGE — censure a droite PAR COHORTE (les extractions finissent a des
# mois differents : 2018->348, 2019->360, 2020->372, 2021+->384). Sans ca, le churn
# saute a ~100% au bord de chaque cohorte.
lo_flux  = flux.groupby("identifiant_tiers")["periode"].max()
lo_stock = stock.groupby("identifiant_tiers")["periode"].max()
last_obs = pd.concat([lo_flux, lo_stock], axis=1).max(axis=1)
coh      = cohort_of.reindex(last_obs.index)
obs_end  = last_obs.groupby(coh).transform("max")        # horizon de donnees de la cohorte
CENSOR_TOL = 1                                            # tolerance latence (mois)

if HAS_UPSTREAM:
    up_first = upstream[upstream["scope"] == "BPL"].groupby("identifiant_tiers")["periode"].min()
else:
    up_first = pd.Series(dtype="float64")
upstreamed = pd.Series(cli.index.isin(up_first.index), index=cli.index)

still_here   = (last_obs >= (obs_end - CENSOR_TOL)).reindex(cli.index, fill_value=False)
churned      = (~still_here) & ~upstreamed
churn_tenure = (last_obs.reindex(cli.index) + 1 - acq.reindex(cli.index)).where(churned)
potential_window = (obs_end.reindex(cli.index) - acq.reindex(cli.index))   # tenure max observable

print("Horizon de donnees par cohorte :")
print(obs_end.groupby(coh).first().to_string())
print(f"\nChurners: {int(churned.sum()):,} | Upstream: {int(upstreamed.sum()):,} | Actifs: {int((~churned & ~upstreamed).sum()):,}")

## Outil — ZOOM sur une seule caracteristique (grand format)

Les vues combinees (facettes) sont pratiques en survol mais illisibles quand une caracteristique
a beaucoup de modalites (ex. CSP). Pour un graphe **grand format sur UNE caracteristique** :
1. re-execute la cellule DataFrame de la metrique voulue (ex. `df_nbi_m3_caract`) ;
2. renseigne les 4 variables ci-dessous ; 3. lance la cellule.
L'ordre des modalites suit `ORDERS` (editable).

In [ ]:
# === ZOOM : un seul graphe pour UNE caracteristique (grand format) ===
Z      = "df_nbi_m3_caract"   # nom du DataFrame '..._caract' a zoomer (entre guillemets)
YCOL   = "nbi_m3"            # colonne de valeur a tracer
CARACT = "csp"              # canal / tranche_revenu / residence / tranche_age / csp
TIME   = "cohorte"          # 'cohorte' (vues M+3) ou 'annee' (vues calendaires)

if Z in dir():
    _z = eval(Z); sub = _z[_z["caracteristique"] == CARACT].copy()
    fig = px.line(sub.astype({TIME: str}), x="categorie", y=YCOL, color=TIME, markers=True,
                  category_orders={"categorie": ORDERS.get(CARACT, sorted(sub["categorie"].astype(str).unique()))},
                  title=f"Zoom — {YCOL} par {CARACT}")
    fig.update_layout(height=560, xaxis_title=CARACT, legend_title=TIME); fig.update_xaxes(tickangle=-40)
    fig.write_html(HTML_DIR / f"zoom_{YCOL}_{CARACT}.html", include_plotlyjs="cdn"); fig.show()
else:
    print(f"Re-execute d'abord la cellule qui construit {Z}, puis relance ce zoom.")

## 1. Qui sont nos nouveaux clients, et comment ca evolue ?

Pour chaque caracteristique : **une cellule DataFrame** puis **un graphe**. L'axe des annees
est la **cohorte d'acquisition** (l'annee ou le client est entre).

### 1.1 Nombre d'acquisitions par annee

In [ ]:
df_acq_annee = (cli.groupby("cohorte").size().reset_index(name="n_acquisitions")
                .sort_values("cohorte"))
df_acq_annee

In [ ]:
fig = px.bar(df_acq_annee, x="cohorte", y="n_acquisitions", text="n_acquisitions",
             title="Nombre d'acquisitions par annee", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_layout(xaxis_title="Annee d'acquisition", yaxis_title="Clients")
save_html(fig, "1_1_acquisitions_par_annee")

### 1.2 Canal (digital vs physique) par annee — volumes et part digitale

In [ ]:
df_canal = (cli.groupby(["cohorte", "canal"]).size().reset_index(name="n"))
df_canal["part_%"] = df_canal.groupby("cohorte")["n"].transform(lambda s: s / s.sum() * 100).round(1)
df_canal.sort_values(["cohorte", "canal"])

In [ ]:
fig = px.bar(df_canal, x="cohorte", y="n", color="canal", barmode="group",
             title="Acquisitions par canal et par annee",
             color_discrete_map={"Digital": "#06B6D4", "Physique": "#94A3B8"})
fig.update_layout(xaxis_title="Annee", yaxis_title="Clients")
save_html(fig, "1_2a_canal_volumes")

In [ ]:
df_digital = df_canal[df_canal["canal"] == "Digital"]
fig = px.line(df_digital, x="cohorte", y="part_%", markers=True,
              title="Part des acquisitions DIGITALES par annee", color_discrete_sequence=["#06B6D4"])
fig.update_layout(xaxis_title="Annee", yaxis_title="% digital"); fig.update_yaxes(range=[0, 100])
save_html(fig, "1_2b_part_digitale")

### 1.2c Combo — volume d'acquisitions (barres) + part digitale (ligne, double axe)

In [ ]:
n = cli.groupby("cohorte").size().rename("n_acquisitions")
p = cli.groupby("cohorte")["canal"].apply(lambda s: (s == "Digital").mean() * 100).round(1).rename("part_digitale_%")
d = pd.concat([n, p], axis=1).reset_index().sort_values("cohorte")
d

In [ ]:
from plotly.subplots import make_subplots
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=d["cohorte"].astype(str), y=d["n_acquisitions"], name="Acquisitions",
            marker_color="#7C3AED", text=d["n_acquisitions"], textposition="outside")
fig.add_scatter(x=d["cohorte"].astype(str), y=d["part_digitale_%"], name="Part digitale (%)",
                mode="lines+markers", line=dict(color="#06B6D4", width=3), secondary_y=True)
fig.update_layout(title="Acquisitions par annee et part digitale", xaxis_title="Annee")
fig.update_yaxes(title_text="Nombre d'acquisitions", secondary_y=False)
fig.update_yaxes(title_text="Part digitale (%)", range=[0, 100], secondary_y=True)
save_html(fig, "1_2c_acquisitions_et_part_digitale"); del d

### 1.3 Age des nouveaux clients par annee

In [ ]:
df_age = (cli.dropna(subset=["age_num"]).groupby("cohorte")["age_num"]
          .agg(age_moyen="mean", age_median="median", n="size").round(1).reset_index())
df_age

In [ ]:
fig = px.line(df_age, x="cohorte", y=["age_moyen", "age_median"], markers=True,
              title="Age moyen / median des nouveaux clients par annee")
fig.update_layout(xaxis_title="Annee", yaxis_title="Age", legend_title="")
save_html(fig, "1_3a_age_evolution")

In [ ]:
fig = px.box(cli.dropna(subset=["age_num"]), x="cohorte", y="age_num",
             title="Distribution de l'age des nouveaux clients par annee", color_discrete_sequence=["#7C3AED"])
fig.update_layout(xaxis_title="Annee", yaxis_title="Age")
save_html(fig, "1_3b_age_distribution")

### 1.3c Distribution des tranches d'age par annee (parts) + age moyen en texte

In [ ]:
df_age_dist = cli.groupby(["cohorte", "tranche_age"], observed=True).size().reset_index(name="n")
df_age_dist["part_%"] = df_age_dist.groupby("cohorte")["n"].transform(lambda s: s / s.sum() * 100).round(1)
age_moy = cli.dropna(subset=["age_num"]).groupby("cohorte")["age_num"].mean().round(1)
print("Age moyen par cohorte :", age_moy.round(0).to_dict())
df_age_dist.head()

In [ ]:
fig = px.bar(df_age_dist, x="cohorte", y="part_%", color="tranche_age", barmode="stack",
             category_orders={"tranche_age": AGE_LABELS}, title="Distribution des tranches d'age par annee (+ age moyen)")
for c, a in age_moy.items():
    fig.add_annotation(x=c, y=103, text=f"{a:.0f} ans", showarrow=False, font=dict(size=10, color="#334155"))
fig.update_layout(xaxis_title="Annee", yaxis_title="% des acquisitions", legend_title="Tranche d'age")
save_html(fig, "1_3c_age_distribution_parts"); del df_age_dist

### 1.4 Tranche de revenu par annee (parts)

In [ ]:
df_rev = (cli.groupby(["cohorte", "tranche_revenu"], observed=True).size().reset_index(name="n"))
df_rev["part_%"] = df_rev.groupby("cohorte")["n"].transform(lambda s: s / s.sum() * 100).round(1)
df_rev.sort_values(["cohorte", "tranche_revenu"])

In [ ]:
fig = px.bar(df_rev, x="cohorte", y="part_%", color="tranche_revenu", barmode="stack",
             title="Repartition des nouveaux clients par tranche de revenu et par annee")
fig.update_layout(xaxis_title="Annee", yaxis_title="% des acquisitions", legend_title="Revenu")
save_html(fig, "1_4_revenu_par_annee")

### 1.5 Residents vs frontaliers par annee

In [ ]:
df_res = (cli.groupby(["cohorte", "residence"]).size().reset_index(name="n"))
df_res["part_%"] = df_res.groupby("cohorte")["n"].transform(lambda s: s / s.sum() * 100).round(1)
df_res.sort_values(["cohorte", "residence"])

In [ ]:
fig = px.bar(df_res, x="cohorte", y="part_%", color="residence", barmode="stack",
             title="Part residents / frontaliers par annee",
             color_discrete_map={"Frontalier": "#EC4899", "Resident": "#06B6D4", "Inconnu": "#CBD5E1"})
fig.update_layout(xaxis_title="Annee", yaxis_title="% des acquisitions", legend_title="Residence")
save_html(fig, "1_5_residence_par_annee")

### 1.6 CSP (segment ameliore) par annee — top modalites

In [ ]:
top_csp = cli["csp"].value_counts().head(6).index.tolist()
df_csp = (cli[cli["csp"].isin(top_csp)].groupby(["cohorte", "csp"]).size().reset_index(name="n"))
df_csp["part_%"] = df_csp.groupby("cohorte")["n"].transform(lambda s: s / s.sum() * 100).round(1)
print("Top CSP:", top_csp); df_csp.head()

In [ ]:
fig = px.bar(df_csp, x="cohorte", y="part_%", color="csp", barmode="stack",
             title="Repartition CSP des nouveaux clients par annee (top 6)")
fig.update_layout(xaxis_title="Annee", yaxis_title="% des acquisitions", legend_title="CSP")
save_html(fig, "1_6_csp_par_annee")

### 1.7 Distribution des CPS par cohorte (snapshot) — tableau

Part (%) de chaque CPS (segment ameliore) dans chaque millesime. Tableau complet (toutes
cohortes) ; pour la slide, selectionne les colonnes voulues (ex. 2019, 2022, 2025).

In [ ]:
tab_csp = (pd.crosstab(cli["csp"], cli["cohorte"], normalize="columns") * 100).round(1)
tab_csp = tab_csp.reindex([c for c in ORDERS["csp"] if c in tab_csp.index])   # ordre par frequence
tab_csp

In [ ]:
fig = px.imshow(tab_csp, text_auto=True, aspect="auto", color_continuous_scale="Greens",
                title="Distribution des CPS par cohorte (%)")
fig.update_layout(xaxis_title="Cohorte", yaxis_title="CPS", height=520, coloraxis_showscale=False)
save_html(fig, "1_7_cps_par_cohorte"); del tab_csp

## 2. NBI — evolution

Deux angles : **(A) annee calendaire** = NBI genere par le book chaque annee civile ;
**(B) cohorte a M+3** = NBI mensuel moyen d'un nouveau client a son 3e mois, par millesime
(qualite du recrutement). Ventilation par **famille de produit**.

### 2.A NBI total par annee calendaire et par famille

In [ ]:
df_nbi_cal = panel.groupby(["annee", "famille"], as_index=False)["nbi"].sum()
df_nbi_cal["nbi_M_EUR"] = (df_nbi_cal["nbi"] / 1e6).round(2)
df_nbi_cal.pivot(index="annee", columns="famille", values="nbi_M_EUR")

In [ ]:
fig = px.line(df_nbi_cal, x="annee", y="nbi_M_EUR", color="famille", markers=True,
              title="NBI total par annee calendaire et par famille", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Annee civile", yaxis_title="NBI (M EUR)", legend_title="Famille")
save_html(fig, "2_A_nbi_calendaire")

#### 2.A bis — NBI moyen par client, decline par caracteristique (vue combinee, couleur = annee)

In [ ]:
nbi_cy = panel.groupby(["identifiant_tiers", "annee"], as_index=False)["nbi"].sum()   # NBI annuel / client
nbi_cy = nbi_cy.merge(cli[CARACTS], left_on="identifiant_tiers", right_index=True, how="inner")
df_nbi_cal_caract = (nbi_cy.melt(id_vars=["annee", "nbi"], value_vars=CARACTS,
                                 var_name="caracteristique", value_name="categorie")
                     .groupby(["caracteristique", "categorie", "annee"], observed=True)["nbi"]
                     .mean().round(0).reset_index())
df_nbi_cal_caract.head()

In [ ]:
fig = px.line(df_nbi_cal_caract.astype({"annee": str}), x="categorie", y="nbi", color="annee",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="NBI annuel moyen par client, par caracteristique et par annee")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Annee")
save_html(fig, "2_A_bis_nbi_par_caract")
del nbi_cy, df_nbi_cal_caract

### 2.B NBI moyen a M+3 par cohorte et par famille

In [ ]:
m3 = panel[panel["tenure"] == 3]
df_nbi_m3 = m3.groupby(["cohorte", "famille"], as_index=False)["nbi"].mean()
df_nbi_m3["nbi_moyen"] = df_nbi_m3["nbi"].round(1)
df_nbi_m3.pivot(index="cohorte", columns="famille", values="nbi_moyen")

In [ ]:
fig = px.line(df_nbi_m3, x="cohorte", y="nbi_moyen", color="famille", markers=True,
              title="NBI mensuel moyen a M+3 par cohorte et par famille", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte d'acquisition", yaxis_title="NBI moyen a M+3 (EUR)", legend_title="Famille")
save_html(fig, "2_B_nbi_m3_cohorte")

#### 2.B bis — NBI moyen a M+3 par client, decline par caracteristique (vue combinee, couleur = cohorte)

In [ ]:
nbi_m3_client = m3.groupby("identifiant_tiers")["nbi"].sum()          # NBI total M+3 par client
base = cli[CARACTS + ["cohorte"]].copy(); base["nbi_m3"] = nbi_m3_client.reindex(cli.index)
df_nbi_m3_caract = (base.melt(id_vars=["nbi_m3", "cohorte"], value_vars=CARACTS,
                              var_name="caracteristique", value_name="categorie")
                    .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["nbi_m3"]
                    .mean().round(1).reset_index())
df_nbi_m3_caract.head()

In [ ]:
fig = px.line(df_nbi_m3_caract.astype({"cohorte": str}), x="categorie", y="nbi_m3", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="NBI moyen a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "2_B_bis_nbi_m3_par_caract")
del df_nbi_m3_caract, base

### 2.C NBI CUMULE a M+3 par cohorte et par famille

Vue (c) a etudier avec la vue (b) : **somme du NBI des 3 premiers mois** (M+0..M+2) par client,
puis moyenne par cohorte. Plus stable que la valeur mensuelle a M+3.

In [ ]:
pnb_cum = early.groupby(["identifiant_tiers", "cohorte", "famille"], as_index=False)["nbi"].sum()
df_pnb_cum_m3 = pnb_cum.groupby(["cohorte", "famille"], as_index=False)["nbi"].mean()
df_pnb_cum_m3["pnb_cumule_moyen"] = df_pnb_cum_m3["nbi"].round(1)
df_pnb_cum_m3.pivot(index="cohorte", columns="famille", values="pnb_cumule_moyen")

In [ ]:
fig = px.line(df_pnb_cum_m3, x="cohorte", y="pnb_cumule_moyen", color="famille", markers=True,
              title="NBI cumule a M+3 (somme des 3 premiers mois) par cohorte et par famille",
              color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="NBI cumule M+3 moyen (EUR)", legend_title="Famille")
save_html(fig, "2_C_nbi_cumule_m3")
del pnb_cum, df_pnb_cum_m3

#### 2.C bis — NBI cumule a M+3 par caracteristique (vue combinee, couleur = cohorte)

In [ ]:
pnb_cum_client = early.groupby("identifiant_tiers")["nbi"].sum()          # NBI cumule total M+3 par client
base = cli[CARACTS + ["cohorte"]].copy(); base["pnb_cum"] = pnb_cum_client.reindex(cli.index)
df_pnb_cum_caract = (base.melt(id_vars=["pnb_cum", "cohorte"], value_vars=CARACTS,
                               var_name="caracteristique", value_name="categorie")
                     .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["pnb_cum"]
                     .mean().round(1).reset_index())
df_pnb_cum_caract.head()

In [ ]:
fig = px.line(df_pnb_cum_caract.astype({"cohorte": str}), x="categorie", y="pnb_cum", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="NBI cumule a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "2_C_bis_nbi_cumule_par_caract")
del df_pnb_cum_caract, base, pnb_cum_client

## 3. Encours — evolution

**(A) annee calendaire** = encours de fin d'annee (book) par famille ;
**(B) cohorte a M+3** = encours moyen des detenteurs a M+3, par millesime.

### 3.A Encours de fin d'annee par famille

In [ ]:
fin_annee = panel.groupby("annee")["periode"].transform("max")
dec = panel[panel["periode"] == fin_annee]                       # dernier mois de chaque annee
df_enc_cal = dec.groupby(["annee", "famille"], as_index=False)["encours"].sum()
df_enc_cal["encours_M_EUR"] = (df_enc_cal["encours"] / 1e6).round(1)
df_enc_cal.pivot(index="annee", columns="famille", values="encours_M_EUR")

In [ ]:
fig = px.line(df_enc_cal, x="annee", y="encours_M_EUR", color="famille", markers=True,
              title="Encours de fin d'annee par famille", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Annee civile", yaxis_title="Encours (M EUR)", legend_title="Famille")
save_html(fig, "3_A_encours_calendaire")

#### 3.A bis — Encours moyen par client (fin d'annee), decline par caracteristique (couleur = annee)

In [ ]:
enc_dec_cy = dec.groupby(["identifiant_tiers", "annee"], as_index=False)["encours"].sum()   # encours fin d'annee / client
enc_dec_cy = enc_dec_cy.merge(cli[CARACTS], left_on="identifiant_tiers", right_index=True, how="inner")
df_enc_cal_caract = (enc_dec_cy.melt(id_vars=["annee", "encours"], value_vars=CARACTS,
                                     var_name="caracteristique", value_name="categorie")
                     .groupby(["caracteristique", "categorie", "annee"], observed=True)["encours"]
                     .mean().round(0).reset_index())
df_enc_cal_caract.head()

In [ ]:
fig = px.line(df_enc_cal_caract.astype({"annee": str}), x="categorie", y="encours", color="annee",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen de fin d'annee par client, par caracteristique et par annee")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Annee")
save_html(fig, "3_A_bis_encours_par_caract")
del enc_dec_cy, df_enc_cal_caract

### 3.B Encours moyen des detenteurs a M+3 par cohorte

In [ ]:
df_enc_m3 = (m3[m3["encours"].fillna(0) > 0].groupby(["cohorte", "famille"], as_index=False)["encours"].mean())
df_enc_m3["encours_moyen"] = df_enc_m3["encours"].round(0)
df_enc_m3.pivot(index="cohorte", columns="famille", values="encours_moyen")

In [ ]:
fig = px.line(df_enc_m3, x="cohorte", y="encours_moyen", color="famille", markers=True,
              title="Encours moyen des detenteurs a M+3 par cohorte", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="Encours moyen a M+3 (EUR)", legend_title="Famille")
save_html(fig, "3_B_encours_m3_cohorte")

#### 3.B bis — Encours moyen a M+3 par client (detenteurs), decline par caracteristique (couleur = cohorte)

In [ ]:
enc_m3_client = m3[m3["encours"].fillna(0) > 0].groupby("identifiant_tiers")["encours"].sum()
base = cli[CARACTS + ["cohorte"]].copy(); base["enc_m3"] = enc_m3_client.reindex(cli.index)
df_enc_m3_caract = (base.dropna(subset=["enc_m3"]).melt(id_vars=["enc_m3", "cohorte"], value_vars=CARACTS,
                        var_name="caracteristique", value_name="categorie")
                    .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc_m3"]
                    .mean().round(0).reset_index())
df_enc_m3_caract.head()

In [ ]:
fig = px.line(df_enc_m3_caract.astype({"cohorte": str}), x="categorie", y="enc_m3", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen a M+3 (detenteurs) par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "3_B_bis_encours_m3_par_caract")
del df_enc_m3_caract, base

### 3.C Encours MOYEN sur 3 mois par cohorte et par famille

Vue (c) : **moyenne de l'encours sur M+0..M+2** (un stock ne se cumule pas), detenteurs seuls.
Moins bruite que le seul snapshot M+3.

In [ ]:
enc_moy = early.groupby(["identifiant_tiers", "cohorte", "famille"], as_index=False)["encours"].mean()
df_enc_moy3 = enc_moy[enc_moy["encours"] > 0].groupby(["cohorte", "famille"], as_index=False)["encours"].mean()
df_enc_moy3["encours_moyen3"] = df_enc_moy3["encours"].round(0)
df_enc_moy3.pivot(index="cohorte", columns="famille", values="encours_moyen3")

In [ ]:
fig = px.line(df_enc_moy3, x="cohorte", y="encours_moyen3", color="famille", markers=True,
              title="Encours moyen sur les 3 premiers mois par cohorte et par famille",
              color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="Encours moyen M+0..M+2 (EUR)", legend_title="Famille")
save_html(fig, "3_C_encours_moyen3")
del enc_moy, df_enc_moy3

#### 3.C bis — Encours moyen sur 3 mois par caracteristique (couleur = cohorte)

In [ ]:
ce = early.groupby(["identifiant_tiers", "tenure"], as_index=False)["encours"].sum()   # total encours par client-mois
enc_moy_client = ce.groupby("identifiant_tiers")["encours"].mean()                       # moyenne sur les 3 mois
base = cli[CARACTS + ["cohorte"]].copy(); base["enc_moy3"] = enc_moy_client.reindex(cli.index)
df_enc_moy3_caract = (base.dropna(subset=["enc_moy3"]).melt(id_vars=["enc_moy3", "cohorte"], value_vars=CARACTS,
                          var_name="caracteristique", value_name="categorie")
                      .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc_moy3"]
                      .mean().round(0).reset_index())
df_enc_moy3_caract.head()

In [ ]:
fig = px.line(df_enc_moy3_caract.astype({"cohorte": str}), x="categorie", y="enc_moy3", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen sur 3 mois par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "3_C_bis_encours_moyen3_par_caract")
del df_enc_moy3_caract, base, ce, enc_moy_client

## 4. Marge (NBI / encours) — evolution

Marge implicite annualisee = `NBI / encours x 12`, calculee **sur les lignes detentrices**
(encours > 0) pour que numerateur et denominateur portent sur les memes observations.
**(A) par annee calendaire**, **(B) a M+3 par cohorte**. Famille **Autre PNB** exclue (pas d'encours).

### 4.A Marge annualisee par famille et par annee calendaire

In [ ]:
h = panel[(panel["encours"].fillna(0) > 0) & (panel["family"].isin(ENC_FAMS))]
df_marge_cal = h.groupby(["annee", "famille"], as_index=False).agg(nbi=("nbi", "sum"), encours=("encours", "sum"))
df_marge_cal["marge_annualisee_%"] = (df_marge_cal["nbi"] / df_marge_cal["encours"] * 12 * 100).round(2)
df_marge_cal.pivot(index="annee", columns="famille", values="marge_annualisee_%")

In [ ]:
fig = px.line(df_marge_cal, x="annee", y="marge_annualisee_%", color="famille", markers=True,
              title="Marge implicite annualisee (NBI/encours) par famille et par annee", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Annee civile", yaxis_title="Marge annualisee (%)", legend_title="Famille")
save_html(fig, "4_A_marge_calendaire")

#### 4.A bis — Marge annualisee par caracteristique et par annee (detenteurs, toutes familles a encours)

In [ ]:
hcy = (panel[(panel["encours"].fillna(0) > 0) & (panel["family"].isin(ENC_FAMS))]
       .groupby(["identifiant_tiers", "annee"], as_index=False).agg(nbi=("nbi", "sum"), enc=("encours", "sum")))
hcy = hcy.merge(cli[CARACTS], left_on="identifiant_tiers", right_index=True, how="inner")
df_marge_cal_caract = (hcy.melt(id_vars=["annee", "nbi", "enc"], value_vars=CARACTS,
                                var_name="caracteristique", value_name="categorie")
                       .groupby(["caracteristique", "categorie", "annee"], observed=True)
                       .agg(nbi=("nbi", "sum"), enc=("enc", "sum")).reset_index())
df_marge_cal_caract["marge_annualisee_%"] = (df_marge_cal_caract["nbi"] / df_marge_cal_caract["enc"] * 12 * 100).round(2)
df_marge_cal_caract.head()

In [ ]:
fig = px.line(df_marge_cal_caract.astype({"annee": str}), x="categorie", y="marge_annualisee_%", color="annee",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Marge annualisee (NBI/encours) par caracteristique et par annee")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Annee")
save_html(fig, "4_A_bis_marge_par_caract")
del hcy, df_marge_cal_caract

### 4.B Marge annualisee a M+3 par cohorte

In [ ]:
h3 = m3[(m3["encours"].fillna(0) > 0) & (m3["family"].isin(ENC_FAMS))]
df_marge_m3 = h3.groupby(["cohorte", "famille"], as_index=False).agg(nbi=("nbi", "sum"), encours=("encours", "sum"))
df_marge_m3["marge_annualisee_%"] = (df_marge_m3["nbi"] / df_marge_m3["encours"] * 12 * 100).round(2)
df_marge_m3.pivot(index="cohorte", columns="famille", values="marge_annualisee_%")

In [ ]:
fig = px.line(df_marge_m3, x="cohorte", y="marge_annualisee_%", color="famille", markers=True,
              title="Marge implicite annualisee a M+3 par cohorte", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="Marge annualisee (%)", legend_title="Famille")
save_html(fig, "4_B_marge_m3_cohorte")

#### 4.B bis — Marge annualisee a M+3 par caracteristique et par cohorte (detenteurs)

In [ ]:
h3c = (m3[(m3["encours"].fillna(0) > 0) & (m3["family"].isin(ENC_FAMS))]
       .groupby("identifiant_tiers", as_index=False).agg(nbi=("nbi", "sum"), enc=("encours", "sum")))
h3c = h3c.merge(cli[CARACTS + ["cohorte"]], left_on="identifiant_tiers", right_index=True, how="inner")
df_marge_m3_caract = (h3c.melt(id_vars=["cohorte", "nbi", "enc"], value_vars=CARACTS,
                               var_name="caracteristique", value_name="categorie")
                      .groupby(["caracteristique", "categorie", "cohorte"], observed=True)
                      .agg(nbi=("nbi", "sum"), enc=("enc", "sum")).reset_index())
df_marge_m3_caract["marge_annualisee_%"] = (df_marge_m3_caract["nbi"] / df_marge_m3_caract["enc"] * 12 * 100).round(2)
df_marge_m3_caract.head()

In [ ]:
fig = px.line(df_marge_m3_caract.astype({"cohorte": str}), x="categorie", y="marge_annualisee_%", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Marge annualisee a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "4_B_bis_marge_m3_par_caract")
del df_marge_m3_caract, h3c

### 4.C Marge sur 3 mois par cohorte et par famille

Vue (c) coherente avec l'encours moyen 3 mois : marge annualisee = `NBI cumule / encours cumule x 12`
sur les lignes detentrices des 3 premiers mois (= NBI cumule / (encours moyen x 3), annualisee).

In [ ]:
hc = early[(early["encours"].fillna(0) > 0) & (early["family"].isin(ENC_FAMS))]
df_marge_c = hc.groupby(["cohorte", "famille"], as_index=False).agg(nbi=("nbi", "sum"), enc=("encours", "sum"))
df_marge_c["marge_annualisee_%"] = (df_marge_c["nbi"] / df_marge_c["enc"] * 12 * 100).round(2)
df_marge_c.pivot(index="cohorte", columns="famille", values="marge_annualisee_%")

In [ ]:
fig = px.line(df_marge_c, x="cohorte", y="marge_annualisee_%", color="famille", markers=True,
              title="Marge annualisee sur 3 mois par cohorte et par famille", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="Marge annualisee (%)", legend_title="Famille")
save_html(fig, "4_C_marge_3mois")
del hc, df_marge_c

#### 4.C bis — Marge sur 3 mois par caracteristique (couleur = cohorte)

In [ ]:
h3c = (early[(early["encours"].fillna(0) > 0) & (early["family"].isin(ENC_FAMS))]
       .groupby("identifiant_tiers", as_index=False).agg(nbi=("nbi", "sum"), enc=("encours", "sum")))
h3c = h3c.merge(cli[CARACTS + ["cohorte"]], left_on="identifiant_tiers", right_index=True, how="inner")
df_marge_c_caract = (h3c.melt(id_vars=["cohorte", "nbi", "enc"], value_vars=CARACTS,
                              var_name="caracteristique", value_name="categorie")
                     .groupby(["caracteristique", "categorie", "cohorte"], observed=True)
                     .agg(nbi=("nbi", "sum"), enc=("enc", "sum")).reset_index())
df_marge_c_caract["marge_annualisee_%"] = (df_marge_c_caract["nbi"] / df_marge_c_caract["enc"] * 12 * 100).round(2)
df_marge_c_caract.head()

In [ ]:
fig = px.line(df_marge_c_caract.astype({"cohorte": str}), x="categorie", y="marge_annualisee_%", color="cohorte",
              markers=True, facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Marge annualisee sur 3 mois par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None)
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "4_C_bis_marge_3mois_par_caract")
del df_marge_c_caract, h3c

## 5. Churn — survie et retention (corrige)

Churn = le client **disparait de la timeline** avant l'horizon de donnees de **sa cohorte**
(censure par cohorte), sauf upstream (BDL→BPL). **(A)** courbe de survie/churn par cohorte ;
**(B)** retention (churn a M+12 / M+24) par millesime.

### 5.A Taux de churn cumule par tenure, une courbe par cohorte

In [ ]:
HORIZ = [3, 6, 12, 18, 24, 36, 48, 60]
rows = []
for c in sorted(cli["cohorte"].dropna().unique()):
    idx = cli.index[cli["cohorte"] == c]
    pw = potential_window.reindex(idx); ct = churn_tenure.reindex(idx)
    for X in HORIZ:
        elig = pw >= X
        if elig.sum() < 20: continue                    # cohorte pas observable a cet horizon
        rows.append({"cohorte": int(c), "tenure": X,
                     "churn_%": round((ct[elig] <= X).mean() * 100, 1), "n_eligibles": int(elig.sum())})
df_churn = pd.DataFrame(rows)
df_churn.pivot(index="tenure", columns="cohorte", values="churn_%")

In [ ]:
fig = px.line(df_churn, x="tenure", y="churn_%", color=df_churn["cohorte"].astype(str), markers=True,
              title="Taux de churn cumule par cohorte (corrige : censure par cohorte)")
fig.update_layout(xaxis_title="Mois depuis l'acquisition (tenure)", yaxis_title="% churn cumule", legend_title="Cohorte")
save_html(fig, "5_A_churn_survie_cohorte")

### 5.B Retention : churn a M+12 et M+24 par millesime

In [ ]:
df_ret = df_churn[df_churn["tenure"].isin([12, 24])].copy()
df_ret["horizon"] = "M+" + df_ret["tenure"].astype(str)
df_ret.pivot(index="cohorte", columns="horizon", values="churn_%")

In [ ]:
fig = px.bar(df_ret, x="cohorte", y="churn_%", color="horizon", barmode="group",
             title="Taux de churn a M+12 et M+24 par millesime d'acquisition",
             color_discrete_map={"M+12": "#F59E0B", "M+24": "#EC4899"})
fig.update_layout(xaxis_title="Cohorte d'acquisition", yaxis_title="% churn")
save_html(fig, "5_B_retention_par_millesime")

### 5.C Tableau churn + upstream cumules par cohorte (Y+1 -> Y+5)

Pour chaque cohorte : % **cumule** de clients **clos (CHURN)** et passes en banque privee **(UP)**
au bout de Y+1..Y+5. `/` = horizon non observable pour cette cohorte (pas assez de recul).

In [ ]:
up_first = (upstream[upstream["scope"] == "BPL"].groupby("identifiant_tiers")["periode"].min()
            if HAS_UPSTREAM else pd.Series(dtype="float64"))
up_tenure = up_first.reindex(cli.index) - acq.reindex(cli.index)

YEARS_T = [1, 2, 3, 4, 5]
rows = {}
for coh in sorted(cli["cohorte"].dropna().unique()):
    idx_all = cli.index[cli["cohorte"] == coh]
    r = {}
    for y in YEARS_T:
        X = 12 * y
        idx = idx_all[potential_window.reindex(idx_all) >= X - 1]     # observable a Y+y
        if len(idx) < 20:
            r[(f"Y+{y}", "CHURN")] = "/"; r[(f"Y+{y}", "UP")] = "/"
        else:
            r[(f"Y+{y}", "CHURN")] = f"{(churn_tenure.reindex(idx) <= X).mean() * 100:.1f} %"
            r[(f"Y+{y}", "UP")]    = f"{(up_tenure.reindex(idx) <= X).mean() * 100:.1f} %"
    rows[int(coh)] = r
tab_cu = pd.DataFrame(rows).T
tab_cu.columns = pd.MultiIndex.from_tuples(tab_cu.columns)
tab_cu.index.name = "Annee d'acquisition"
tab_cu

In [ ]:
# Lecture (cohorte la plus ancienne disponible)
c0 = int(sorted(cli["cohorte"].dropna().unique())[0])
idx0 = cli.index[cli["cohorte"] == c0]
print(f"Lecture — clients acquis {c0} :")
for y in [1, 2]:
    X = 12 * y
    ch = (churn_tenure.reindex(idx0) <= X).mean() * 100
    up = (up_tenure.reindex(idx0) <= X).mean() * 100
    print(f"  {y} an(s) apres l'EER : {ch:.0f}% clos, {up:.0f}% upstream, {100-ch-up:.0f}% encore dans le parc")
del up_first, up_tenure

## 6. Activite sur les premiers mois — equipement & encours par produit

Taux d'equipement et encours moyens a **M+3 / M+6 / M+12** par produit (sur toutes les cohortes,
clients encore la ou non), puis declinaison par caracteristique. Enfin : equipement des clients
qui **churnent** (juste avant depart) et qui **upstreament**, et trajectoire depuis l'equipement initial.

In [ ]:
EQ_HOR = [3, 6, 12]
COHORTS = sorted(cli["cohorte"].dropna().unique())
# Detenteurs (encours>0 ou nbi>0) par famille a chaque horizon
held = {t: panel[(panel["tenure"] == t) & ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]
              .groupby("family")["identifiant_tiers"].apply(set).to_dict() for t in EQ_HOR}
print("Detenteurs calcules pour M+3/6/12 :", {t: {f: len(s) for f, s in held[t].items()} for t in EQ_HOR})

### 6.1 Taux d'equipement par produit, par horizon et par cohorte (base)

In [ ]:
recs = []
for t in EQ_HOR:
    for fam in FAM_ORDER:
        hs = held[t].get(fam, set())
        for coh in COHORTS:
            idx = cli.index[(cli["cohorte"] == coh) & (potential_window.reindex(cli.index) >= t)]
            if len(idx) < 10:
                continue
            recs.append({"famille": FAM_LABELS[fam], "horizon": f"M+{t}", "cohorte": int(coh),
                         "taux_%": round(len(set(idx) & hs) / len(idx) * 100, 1)})
df_equip = pd.DataFrame(recs)
df_equip.pivot_table(index=["cohorte"], columns=["famille", "horizon"], values="taux_%").head()

In [ ]:
fig = px.line(df_equip, x="cohorte", y="taux_%", color="famille", facet_col="horizon", markers=True,
              color_discrete_map=FAM_COLOR, category_orders={"horizon": ["M+3", "M+6", "M+12"]},
              title="Taux d'equipement par produit, par horizon et par cohorte")
strip_facet_titles(fig); fig.update_layout(height=450, legend_title="Famille"); fig.update_yaxes(title="% equipes")
save_html(fig, "6_1_equipement_base"); del df_equip

#### 6.2 — Taux d'equipement Epargne a M+3 par caracteristique (couleur = cohorte)

In [ ]:
hs = held[3].get("epargne", set())
base = cli[CARACTS + ["cohorte"]].copy()
base["holds"] = cli.index.isin(hs).astype(float)
base = base[potential_window.reindex(cli.index) >= 3]
df_e = (base.melt(id_vars=["holds", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["holds"].mean().mul(100).round(1).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="holds", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Taux d'equipement Epargne a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="% equipes")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_2_equip_epargne_par_caract"); del df_e, base

#### 6.2 — Taux d'equipement Investissements a M+3 par caracteristique (couleur = cohorte)

In [ ]:
hs = held[3].get("invest", set())
base = cli[CARACTS + ["cohorte"]].copy()
base["holds"] = cli.index.isin(hs).astype(float)
base = base[potential_window.reindex(cli.index) >= 3]
df_e = (base.melt(id_vars=["holds", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["holds"].mean().mul(100).round(1).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="holds", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Taux d'equipement Investissements a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="% equipes")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_2_equip_invest_par_caract"); del df_e, base

#### 6.2 — Taux d'equipement Credit immo a M+3 par caracteristique (couleur = cohorte)

In [ ]:
hs = held[3].get("credit_immo", set())
base = cli[CARACTS + ["cohorte"]].copy()
base["holds"] = cli.index.isin(hs).astype(float)
base = base[potential_window.reindex(cli.index) >= 3]
df_e = (base.melt(id_vars=["holds", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["holds"].mean().mul(100).round(1).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="holds", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Taux d'equipement Credit immo a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="% equipes")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_2_equip_credit_immo_par_caract"); del df_e, base

#### 6.2 — Taux d'equipement Autres credits a M+3 par caracteristique (couleur = cohorte)

In [ ]:
hs = held[3].get("autres_cr", set())
base = cli[CARACTS + ["cohorte"]].copy()
base["holds"] = cli.index.isin(hs).astype(float)
base = base[potential_window.reindex(cli.index) >= 3]
df_e = (base.melt(id_vars=["holds", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["holds"].mean().mul(100).round(1).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="holds", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Taux d'equipement Autres credits a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="% equipes")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_2_equip_autres_cr_par_caract"); del df_e, base

#### 6.2 — Taux d'equipement Autre PNB a M+3 par caracteristique (couleur = cohorte)

In [ ]:
hs = held[3].get("cartes", set())
base = cli[CARACTS + ["cohorte"]].copy()
base["holds"] = cli.index.isin(hs).astype(float)
base = base[potential_window.reindex(cli.index) >= 3]
df_e = (base.melt(id_vars=["holds", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["holds"].mean().mul(100).round(1).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="holds", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Taux d'equipement Autre PNB a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="% equipes")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_2_equip_cartes_par_caract"); del df_e, base

### 6.3 Encours moyen des detenteurs par produit, par horizon et par cohorte (base)

In [ ]:
frames = []
for t in EQ_HOR:
    sub = panel[(panel["tenure"] == t) & (panel["family"].isin(ENC_FAMS)) & (panel["encours"].fillna(0) > 0)]
    g = sub.groupby(["cohorte", "famille"], as_index=False)["encours"].mean()
    g["horizon"] = f"M+{t}"
    frames.append(g)
df_enc_hor = pd.concat(frames, ignore_index=True); df_enc_hor["encours"] = df_enc_hor["encours"].round(0)
df_enc_hor.pivot_table(index="cohorte", columns=["famille", "horizon"], values="encours").head()

In [ ]:
fig = px.line(df_enc_hor, x="cohorte", y="encours", color="famille", facet_col="horizon", markers=True,
              color_discrete_map=FAM_COLOR, category_orders={"horizon": ["M+3", "M+6", "M+12"]},
              title="Encours moyen des detenteurs par produit, par horizon et par cohorte")
strip_facet_titles(fig); fig.update_layout(height=450, legend_title="Famille"); fig.update_yaxes(title="Encours moyen (EUR)")
save_html(fig, "6_3_encours_horizon_base"); del df_enc_hor, frames

#### 6.4 — Encours moyen Epargne a M+3 par caracteristique (detenteurs, couleur = cohorte)

In [ ]:
enc_c = (panel[(panel["family"] == "epargne") & (panel["tenure"] == 3) & (panel["encours"].fillna(0) > 0)]
         .groupby("identifiant_tiers")["encours"].sum())
base = cli[CARACTS + ["cohorte"]].copy(); base["enc"] = enc_c.reindex(cli.index)
df_e = (base.dropna(subset=["enc"]).melt(id_vars=["enc", "cohorte"], value_vars=CARACTS,
            var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc"].mean().round(0).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="enc", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen Epargne a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Encours moyen (EUR)")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_4_encours_epargne_par_caract"); del df_e, base, enc_c

#### 6.4 — Encours moyen Investissements a M+3 par caracteristique (detenteurs, couleur = cohorte)

In [ ]:
enc_c = (panel[(panel["family"] == "invest") & (panel["tenure"] == 3) & (panel["encours"].fillna(0) > 0)]
         .groupby("identifiant_tiers")["encours"].sum())
base = cli[CARACTS + ["cohorte"]].copy(); base["enc"] = enc_c.reindex(cli.index)
df_e = (base.dropna(subset=["enc"]).melt(id_vars=["enc", "cohorte"], value_vars=CARACTS,
            var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc"].mean().round(0).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="enc", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen Investissements a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Encours moyen (EUR)")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_4_encours_invest_par_caract"); del df_e, base, enc_c

#### 6.4 — Encours moyen Credit immo a M+3 par caracteristique (detenteurs, couleur = cohorte)

In [ ]:
enc_c = (panel[(panel["family"] == "credit_immo") & (panel["tenure"] == 3) & (panel["encours"].fillna(0) > 0)]
         .groupby("identifiant_tiers")["encours"].sum())
base = cli[CARACTS + ["cohorte"]].copy(); base["enc"] = enc_c.reindex(cli.index)
df_e = (base.dropna(subset=["enc"]).melt(id_vars=["enc", "cohorte"], value_vars=CARACTS,
            var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc"].mean().round(0).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="enc", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen Credit immo a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Encours moyen (EUR)")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_4_encours_credit_immo_par_caract"); del df_e, base, enc_c

#### 6.4 — Encours moyen Autres credits a M+3 par caracteristique (detenteurs, couleur = cohorte)

In [ ]:
enc_c = (panel[(panel["family"] == "autres_cr") & (panel["tenure"] == 3) & (panel["encours"].fillna(0) > 0)]
         .groupby("identifiant_tiers")["encours"].sum())
base = cli[CARACTS + ["cohorte"]].copy(); base["enc"] = enc_c.reindex(cli.index)
df_e = (base.dropna(subset=["enc"]).melt(id_vars=["enc", "cohorte"], value_vars=CARACTS,
            var_name="caracteristique", value_name="categorie")
        .groupby(["caracteristique", "categorie", "cohorte"], observed=True)["enc"].mean().round(0).reset_index())
df_e.head()

In [ ]:
fig = px.line(df_e.astype({"cohorte": str}), x="categorie", y="enc", color="cohorte", markers=True,
              facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
              title="Encours moyen Autres credits a M+3 par caracteristique et par cohorte")
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Encours moyen (EUR)")
strip_facet_titles(fig); fig.update_layout(height=650, legend_title="Cohorte")
save_html(fig, "6_4_encours_autres_cr_par_caract"); del df_e, base, enc_c

### 6.5 Equipement & encours des clients qui churnent, juste avant de partir

Etat au **dernier mois actif** (tenure = churn_tenure - 1). Ces clients partaient-ils deja
peu equipes / avec peu d'encours ?

In [ ]:
own_ten = (churn_tenure - 1).dropna().astype(int)
pv = panel.merge(own_ten.rename("t"), left_on="identifiant_tiers", right_index=True)
pv = pv[pv["tenure"] == pv["t"]]
eq_churn = pv[(pv["encours"].fillna(0) > 0) | (pv["nbi"].fillna(0) > 0)].groupby("identifiant_tiers")["family"].nunique()
enc_churn = pv.groupby("identifiant_tiers")["encours"].sum()
churn_idx = cli.index[churned]
base = cli.loc[churn_idx, CARACTS + ["cohorte"]].copy()
base["nb_produits"] = eq_churn.reindex(churn_idx); base["encours"] = enc_churn.reindex(churn_idx)
df_churn_eq = (base.melt(id_vars=["nb_produits", "encours", "cohorte"], value_vars=CARACTS,
                         var_name="caracteristique", value_name="categorie")
               .groupby(["caracteristique", "categorie"], observed=True)
               .agg(nb_produits=("nb_produits", "mean"), encours=("encours", "median")).round(1).reset_index())
del pv; df_churn_eq.head()

In [ ]:
fig = px.bar(df_churn_eq, x="categorie", y="nb_produits", facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
             title="Nombre de produits detenus au dernier mois, clients qui churnent", color_discrete_sequence=["#EC4899"])
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Nb produits")
strip_facet_titles(fig); fig.update_layout(height=650)
save_html(fig, "6_5_churn_equipement"); del df_churn_eq, base, eq_churn, enc_churn

### 6.6 Equipement des clients qui upstreament (nb de produits a M+3)

In [ ]:
eq_m3 = (panel[(panel["tenure"] == 3) & ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]
         .groupby("identifiant_tiers")["family"].nunique())
up_idx = cli.index[upstreamed]
base = cli.loc[up_idx, CARACTS + ["cohorte"]].copy(); base["nb_produits"] = eq_m3.reindex(up_idx)
df_up_eq = (base.melt(id_vars=["nb_produits", "cohorte"], value_vars=CARACTS, var_name="caracteristique", value_name="categorie")
            .groupby(["caracteristique", "categorie"], observed=True)["nb_produits"].mean().round(2).reset_index())
print(f"Upstreamers: {len(up_idx):,} | nb produits moyen M+3: {eq_m3.reindex(up_idx).mean():.2f} "
      f"(vs {eq_m3.reindex(cli.index).mean():.2f} tous)"); df_up_eq.head()

In [ ]:
fig = px.bar(df_up_eq, x="categorie", y="nb_produits", facet_col="caracteristique", facet_col_wrap=3, category_orders={"categorie": UNION_ORDER},
             title="Nombre de produits a M+3 des clients qui upstreament", color_discrete_sequence=["#84CC16"])
fig.update_xaxes(matches=None, showticklabels=True); fig.update_yaxes(matches=None, title="Nb produits M+3")
strip_facet_titles(fig); fig.update_layout(height=650)
save_html(fig, "6_6_upstream_equipement"); del df_up_eq, base, eq_m3

### 6.7 Trajectoire d'encours depuis l'equipement initial

Clients equipes d'un produit des M+0/M+1 : comment leur encours evolue-t-il ensuite, par cohorte ?

In [ ]:
TMAX = 60
frames = []
for fam in ENC_FAMS:
    earlyset = set(panel[(panel["family"] == fam) & (panel["tenure"] <= 1) & (panel["encours"].fillna(0) > 0)]["identifiant_tiers"])
    sub = panel[(panel["family"] == fam) & (panel["identifiant_tiers"].isin(earlyset)) &
                (panel["tenure"].between(0, TMAX)) & (panel["encours"].fillna(0) > 0)]
    frames.append(sub.groupby(["cohorte", "famille", "tenure"], as_index=False)["encours"].mean())
df_traj = pd.concat(frames, ignore_index=True)
df_traj.head()

In [ ]:
fig = px.line(df_traj.astype({"cohorte": str}), x="tenure", y="encours", color="cohorte",
              facet_col="famille", facet_col_wrap=2,
              title="Trajectoire d'encours depuis l'equipement initial (equipes des M+0/M+1)")
fig.update_xaxes(matches=None); fig.update_yaxes(matches=None, title="Encours moyen (EUR)")
strip_facet_titles(fig); fig.update_layout(height=700, legend_title="Cohorte")
save_html(fig, "6_7_trajectoire_initiale"); del df_traj, frames

## 7. Comment calibrer le modele ?

Persistance de l'equipement, des encours et du PnB entre M+3, M+12 et Y5 (M+60) ;
marge par produit par annee ; niveau des deciles d'encours par produit par annee.
Correlations = Spearman (robuste a la longue traine).

### 7.1 Retention de l'equipement : M+3 -> M+12 et M+12 -> Y5 (%)

In [ ]:
CAL_HELD = {t: panel[(panel["tenure"] == t) & ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]
                  .groupby("family")["identifiant_tiers"].apply(set).to_dict() for t in [3, 12, 60]}
e12 = set(cli.index[potential_window.reindex(cli.index) >= 12])
e60 = set(cli.index[potential_window.reindex(cli.index) >= 60])
rows = []
for fam in FAM_ORDER:
    h3, h12, h60 = CAL_HELD[3].get(fam, set()), CAL_HELD[12].get(fam, set()), CAL_HELD[60].get(fam, set())
    b1, b2 = (h3 & e12), (h12 & e60)
    rows.append({"famille": FAM_LABELS[fam],
                 "M+3 -> M+12": round(len(b1 & h12) / max(len(b1), 1) * 100, 1),
                 "M+12 -> Y5": round(len(b2 & h60) / max(len(b2), 1) * 100, 1)})
df_ret_eq = pd.DataFrame(rows)
df_ret_eq

In [ ]:
d = df_ret_eq.melt(id_vars="famille", var_name="transition", value_name="retention_%")
fig = px.bar(d, x="famille", y="retention_%", color="transition", barmode="group",
             title="Retention de l'equipement par produit", color_discrete_map={"M+3 -> M+12": "#06B6D4", "M+12 -> Y5": "#7C3AED"})
fig.update_layout(yaxis_title="% encore detenteurs", xaxis_title="")
save_html(fig, "7_1_retention_equipement"); del df_ret_eq, d

### 7.2 Correlation des encours : M+3 vs M+12 et M+12 vs Y5, par produit (Spearman)

In [ ]:
rows = []
for fam in ENC_FAMS:
    e3  = panel[(panel["family"] == fam) & (panel["tenure"] == 3)].groupby("identifiant_tiers")["encours"].sum()
    e12 = panel[(panel["family"] == fam) & (panel["tenure"] == 12)].groupby("identifiant_tiers")["encours"].sum()
    e60 = panel[(panel["family"] == fam) & (panel["tenure"] == 60)].groupby("identifiant_tiers")["encours"].sum()
    d1 = pd.concat([e3.rename("a"), e12.rename("b")], axis=1).dropna(); d1 = d1[d1["a"] > 0]
    d2 = pd.concat([e12.rename("a"), e60.rename("b")], axis=1).dropna(); d2 = d2[d2["a"] > 0]
    rows.append({"famille": FAM_LABELS[fam],
                 "M+3 vs M+12": round(d1.corr(method="spearman").iloc[0, 1], 3) if len(d1) > 30 else np.nan, "n1": len(d1),
                 "M+12 vs Y5": round(d2.corr(method="spearman").iloc[0, 1], 3) if len(d2) > 30 else np.nan, "n2": len(d2)})
df_enc_corr = pd.DataFrame(rows)
df_enc_corr

In [ ]:
d = df_enc_corr.melt(id_vars="famille", value_vars=["M+3 vs M+12", "M+12 vs Y5"], var_name="paire", value_name="spearman")
fig = px.bar(d, x="famille", y="spearman", color="paire", barmode="group",
             title="Correlation des encours (Spearman) par produit", color_discrete_map={"M+3 vs M+12": "#06B6D4", "M+12 vs Y5": "#EC4899"})
fig.update_layout(yaxis_title="Spearman", xaxis_title=""); fig.update_yaxes(range=[0, 1])
save_html(fig, "7_2_correlation_encours"); del df_enc_corr, d

### 7.3 Correlation du PnB cumule : M+3 vs M+12 et M+12 vs Y5 (client total, Spearman)

In [ ]:
c3  = panel[panel["tenure"] < 3].groupby("identifiant_tiers")["nbi"].sum()
c12 = panel[panel["tenure"] < 12].groupby("identifiant_tiers")["nbi"].sum()
c60 = panel[panel["tenure"] < 60].groupby("identifiant_tiers")["nbi"].sum()
d1 = pd.concat([c3.rename("a"), c12.rename("b")], axis=1).dropna()
d2 = pd.concat([c12.rename("a"), c60.rename("b")], axis=1).dropna()
df_pnb_corr = pd.DataFrame([{"paire": "M+3 vs M+12", "spearman": round(d1.corr(method="spearman").iloc[0, 1], 3), "n": len(d1)},
                            {"paire": "M+12 vs Y5", "spearman": round(d2.corr(method="spearman").iloc[0, 1], 3), "n": len(d2)}])
del c3, c12, c60, d1, d2; df_pnb_corr

In [ ]:
fig = px.bar(df_pnb_corr, x="paire", y="spearman", text="spearman", color="paire",
             title="Correlation du PnB cumule (client total, Spearman)", color_discrete_sequence=["#7C3AED", "#F59E0B"])
fig.update_traces(textposition="outside"); fig.update_yaxes(range=[0, 1]); fig.update_layout(showlegend=False, xaxis_title="")
save_html(fig, "7_3_correlation_pnb"); del df_pnb_corr

### 7.4 Marge annualisee par produit par annee (reference de calibration)

In [ ]:
hh = panel[(panel["encours"].fillna(0) > 0) & (panel["family"].isin(ENC_FAMS))]
df_marge_grid = hh.groupby(["famille", "annee"], as_index=False).agg(nbi=("nbi", "sum"), enc=("encours", "sum"))
df_marge_grid["marge_%"] = (df_marge_grid["nbi"] / df_marge_grid["enc"] * 12 * 100).round(2)
piv = df_marge_grid.pivot(index="famille", columns="annee", values="marge_%")
del hh; piv

In [ ]:
fig = px.imshow(piv, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                title="Marge annualisee (%) par produit et par annee")
fig.update_layout(xaxis_title="Annee", yaxis_title="")
save_html(fig, "7_4_marge_grille"); del df_marge_grid, piv

### 7.5 Niveau des deciles d'encours par produit par annee (detenteurs)

In [ ]:
DEC = [0.1, 0.25, 0.5, 0.75, 0.9]
rows = []
for fam in ENC_FAMS:
    for yr in sorted(panel["annee"].unique()):
        v = panel[(panel["family"] == fam) & (panel["annee"] == yr) & (panel["encours"].fillna(0) > 0)]["encours"]
        if len(v) < 50:
            continue
        for q in DEC:
            rows.append({"famille": FAM_LABELS[fam], "annee": yr, "decile": f"D{int(q*100)}", "valeur": round(v.quantile(q), 0)})
df_dec = pd.DataFrame(rows)
df_dec.head()

In [ ]:
fig = px.line(df_dec, x="annee", y="valeur", color="decile", facet_col="famille", facet_col_wrap=2, markers=True,
              title="Deciles d'encours (detenteurs) par produit et par annee")
fig.update_xaxes(matches=None); fig.update_yaxes(matches=None, title="Encours (EUR)")
strip_facet_titles(fig); fig.update_layout(height=700, legend_title="Decile")
save_html(fig, "7_5_deciles_encours"); del df_dec

## 8. Role du credit immobilier

Deux angles : **(A) ancrage / retention** — les detenteurs de credit immo churnent-ils moins,
restent-ils plus longtemps et generent-ils plus de valeur ? ; **(B) contribution / evolution** —
part du credit immo dans le NBI et les encours, penetration par cohorte.

> Detenteur = client ayant eu un encours credit immo > 0 a un moment de la fenetre observee.

In [ ]:
immo_holders = set(panel[(panel["family"] == "credit_immo") & (panel["encours"].fillna(0) > 0)]["identifiant_tiers"])
cli["immo"] = np.where(cli.index.isin(immo_holders), "Detenteur immo", "Sans immo")
print(cli["immo"].value_counts().to_string())

### 8.A.1 Taux de churn : detenteurs de credit immo vs autres

In [ ]:
rows = []
for X in [12, 24, 36]:
    for grp in ["Detenteur immo", "Sans immo"]:
        idx = cli.index[(cli["immo"] == grp) & (potential_window.reindex(cli.index) >= X)]
        if len(idx) < 20:
            continue
        rows.append({"horizon": f"M+{X}", "groupe": grp, "churn_%": round((churn_tenure.reindex(idx) <= X).mean() * 100, 1), "n": len(idx)})
df_immo_churn = pd.DataFrame(rows)
df_immo_churn

In [ ]:
fig = px.bar(df_immo_churn, x="horizon", y="churn_%", color="groupe", barmode="group",
             title="Taux de churn selon la detention de credit immo",
             color_discrete_map={"Detenteur immo": "#EC4899", "Sans immo": "#94A3B8"})
fig.update_layout(yaxis_title="% churn", xaxis_title="")
save_html(fig, "8_A1_churn_immo"); del df_immo_churn

### 8.A.2 Survie (churn cumule) par tenure : detenteurs immo vs autres

In [ ]:
rows = []
for X in [3, 6, 12, 18, 24, 36, 48, 60]:
    for grp in ["Detenteur immo", "Sans immo"]:
        idx = cli.index[(cli["immo"] == grp) & (potential_window.reindex(cli.index) >= X)]
        if len(idx) < 20:
            continue
        rows.append({"tenure": X, "groupe": grp, "churn_%": round((churn_tenure.reindex(idx) <= X).mean() * 100, 1)})
df_surv_immo = pd.DataFrame(rows)
df_surv_immo.pivot(index="tenure", columns="groupe", values="churn_%")

In [ ]:
fig = px.line(df_surv_immo, x="tenure", y="churn_%", color="groupe", markers=True,
              title="Churn cumule par tenure selon la detention de credit immo",
              color_discrete_map={"Detenteur immo": "#EC4899", "Sans immo": "#94A3B8"})
fig.update_layout(xaxis_title="Mois depuis l'acquisition", yaxis_title="% churn cumule")
save_html(fig, "8_A2_survie_immo"); del df_surv_immo

### 8.A.3 Valeur & equipement : detenteurs immo vs autres

In [ ]:
client_nbi = flux.groupby("identifiant_tiers")["nbi"].sum()
nb_prod = (panel[(panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0)]
           .groupby("identifiant_tiers")["family"].nunique())
base = cli[["immo"]].copy()
base["nbi_total"] = client_nbi.reindex(cli.index); base["nb_produits"] = nb_prod.reindex(cli.index)
df_immo_val = base.groupby("immo").agg(nbi_total_moyen=("nbi_total", "mean"),
                                       nb_produits_moyen=("nb_produits", "mean"), n=("immo", "size")).round(2).reset_index()
del base; df_immo_val

In [ ]:
fig = px.bar(df_immo_val, x="immo", y="nbi_total_moyen", text="nbi_total_moyen", color="immo",
             title="NBI total moyen par client selon la detention de credit immo",
             color_discrete_map={"Detenteur immo": "#EC4899", "Sans immo": "#94A3B8"})
fig.update_traces(textposition="outside"); fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="NBI total moyen (EUR)")
save_html(fig, "8_A3a_nbi_immo")

In [ ]:
fig = px.bar(df_immo_val, x="immo", y="nb_produits_moyen", text="nb_produits_moyen", color="immo",
             title="Nombre de produits moyen selon la detention de credit immo",
             color_discrete_map={"Detenteur immo": "#EC4899", "Sans immo": "#94A3B8"})
fig.update_traces(textposition="outside"); fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Nb produits moyen")
save_html(fig, "8_A3b_produits_immo"); del df_immo_val, client_nbi, nb_prod

### 8.B.1 Part du credit immo dans le NBI et les encours, par annee

In [ ]:
tot_nbi = panel.groupby("annee")["nbi"].sum()
immo_nbi = panel[panel["family"] == "credit_immo"].groupby("annee")["nbi"].sum()
fin_annee = panel.groupby("annee")["periode"].transform("max"); dec = panel[panel["periode"] == fin_annee]
tot_enc = dec.groupby("annee")["encours"].sum()
immo_enc = dec[dec["family"] == "credit_immo"].groupby("annee")["encours"].sum()
df_immo_part = pd.DataFrame({"part_nbi_%": (immo_nbi / tot_nbi * 100).round(1),
                             "part_encours_%": (immo_enc / tot_enc * 100).round(1)}).reset_index()
del dec; df_immo_part

In [ ]:
d = df_immo_part.melt(id_vars="annee", var_name="mesure", value_name="part_%")
fig = px.line(d, x="annee", y="part_%", color="mesure", markers=True,
              title="Part du credit immo dans le NBI et les encours, par annee")
fig.update_layout(xaxis_title="Annee", yaxis_title="% du total", legend_title="")
save_html(fig, "8_B1_part_immo"); del df_immo_part, d

### 8.B.2 Penetration du credit immo par cohorte d'acquisition

In [ ]:
rows = [{"cohorte": int(c), "penetration_%": round(cli.loc[cli["cohorte"] == c, "immo"].eq("Detenteur immo").mean() * 100, 1)}
        for c in COHORTS]
df_immo_pen = pd.DataFrame(rows)
df_immo_pen

In [ ]:
fig = px.bar(df_immo_pen, x="cohorte", y="penetration_%", text="penetration_%",
             title="Penetration du credit immo par cohorte d'acquisition", color_discrete_sequence=["#EC4899"])
fig.update_traces(textposition="outside"); fig.update_layout(xaxis_title="Cohorte", yaxis_title="% detenteurs")
save_html(fig, "8_B2_penetration_immo"); del df_immo_pen

## 9. PNB cumule sur 5 ans — le KPI cible

Le modele vise le **PNB genere sur les 5 premieres annees** (60 mois, tenure 0..59) d'un
nouveau client — churn inclus (les mois apres depart valent 0, donc c'est la valeur *reelle*).
On le calcule sur les **cohortes qui ont 5 ans de recul** (clients dont la fenetre >= 59 mois,
soit surtout les acquisitions de debut 2018-2021), puis on donne l'ODG global et la ventilation
par caracteristique (bar par caract) et par famille.

In [ ]:
H5 = 60                                                           # 5 ans = 60 mois (tenure 0..59)
elig5 = cli.index[potential_window.reindex(cli.index) >= H5 - 1]   # clients avec 5 ans complets
# PNB total = NBI total client depuis FLUX (source fiable ; le panel produit peut etre creux, ex. 2018)
pnb5 = flux[flux["tenure"] < H5].groupby("identifiant_tiers")["nbi"].sum()
cli_5y = cli.loc[elig5].copy()
cli_5y["pnb_5y"] = pnb5.reindex(elig5).fillna(0)
print(f"Clients avec 5 ans d'historique : {len(cli_5y):,}")
print(f"Cohortes concernees : {sorted(cli_5y['cohorte'].dropna().unique())}")
print(f"\n*** ODG : un nouveau client genere en moyenne {cli_5y['pnb_5y'].mean():,.0f} EUR de PNB sur 5 ans ***")
print(f"    (median {cli_5y['pnb_5y'].median():,.0f} | total {cli_5y['pnb_5y'].sum()/1e6:,.1f} M EUR)")

### 9.1 Distribution du PNB 5 ans par client

In [ ]:
v = cli_5y["pnb_5y"]; vc = v.clip(v.quantile(0.01), v.quantile(0.99))
fig = px.histogram(vc, nbins=60, title="Distribution du PNB cumule sur 5 ans par client (clip 1-99%)",
                   color_discrete_sequence=["#334155"])
fig.add_vline(x=v.mean(), line_dash="dash", line_color="#EC4899", annotation_text=f"Moyenne {v.mean():,.0f}")
fig.add_vline(x=v.median(), line_dash="dash", line_color="#F59E0B", annotation_text=f"Mediane {v.median():,.0f}")
fig.update_layout(xaxis_title="PNB 5 ans (EUR)", yaxis_title="Nb clients", showlegend=False)
save_html(fig, "9_1_distribution_pnb5")

### 9.2 — PNB 5 ans moyen par canal (agrege, cohortes 5 ans)

In [ ]:
df_p5 = (cli_5y.groupby("canal", observed=True)["pnb_5y"].mean()
         .reindex(ORDERS["canal"]).dropna().round(0).reset_index())
df_p5

In [ ]:
fig = px.bar(df_p5, x="canal", y="pnb_5y", text="pnb_5y",
             title="PNB moyen sur 5 ans par canal", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_xaxes(tickangle=-30)
fig.update_layout(xaxis_title="canal", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_2_pnb5_canal_agrege"); del df_p5

### 9.2 — PNB 5 ans moyen par tranche_revenu (agrege, cohortes 5 ans)

In [ ]:
df_p5 = (cli_5y.groupby("tranche_revenu", observed=True)["pnb_5y"].mean()
         .reindex(ORDERS["tranche_revenu"]).dropna().round(0).reset_index())
df_p5

In [ ]:
fig = px.bar(df_p5, x="tranche_revenu", y="pnb_5y", text="pnb_5y",
             title="PNB moyen sur 5 ans par tranche_revenu", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_xaxes(tickangle=-30)
fig.update_layout(xaxis_title="tranche_revenu", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_2_pnb5_tranche_revenu_agrege"); del df_p5

### 9.2 — PNB 5 ans moyen par residence (agrege, cohortes 5 ans)

In [ ]:
df_p5 = (cli_5y.groupby("residence", observed=True)["pnb_5y"].mean()
         .reindex(ORDERS["residence"]).dropna().round(0).reset_index())
df_p5

In [ ]:
fig = px.bar(df_p5, x="residence", y="pnb_5y", text="pnb_5y",
             title="PNB moyen sur 5 ans par residence", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_xaxes(tickangle=-30)
fig.update_layout(xaxis_title="residence", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_2_pnb5_residence_agrege"); del df_p5

### 9.2 — PNB 5 ans moyen par tranche_age (agrege, cohortes 5 ans)

In [ ]:
df_p5 = (cli_5y.groupby("tranche_age", observed=True)["pnb_5y"].mean()
         .reindex(ORDERS["tranche_age"]).dropna().round(0).reset_index())
df_p5

In [ ]:
fig = px.bar(df_p5, x="tranche_age", y="pnb_5y", text="pnb_5y",
             title="PNB moyen sur 5 ans par tranche_age", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_xaxes(tickangle=-30)
fig.update_layout(xaxis_title="tranche_age", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_2_pnb5_tranche_age_agrege"); del df_p5

### 9.2 — PNB 5 ans moyen par csp (agrege, cohortes 5 ans)

In [ ]:
df_p5 = (cli_5y.groupby("csp", observed=True)["pnb_5y"].mean()
         .reindex(ORDERS["csp"]).dropna().round(0).reset_index())
df_p5

In [ ]:
fig = px.bar(df_p5, x="csp", y="pnb_5y", text="pnb_5y",
             title="PNB moyen sur 5 ans par csp", color_discrete_sequence=["#7C3AED"])
fig.update_traces(textposition="outside"); fig.update_xaxes(tickangle=-30)
fig.update_layout(xaxis_title="csp", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_2_pnb5_csp_agrege"); del df_p5

### 9.3 — PNB 5 ans moyen par canal et par cohorte

In [ ]:
df_p5c = (cli_5y.groupby(["cohorte", "canal"], observed=True)["pnb_5y"].mean().round(0).reset_index())
df_p5c.head()

In [ ]:
fig = px.bar(df_p5c.astype({"cohorte": str}), x="canal", y="pnb_5y", color="cohorte", barmode="group",
             category_orders={"canal": ORDERS["canal"]}, title="PNB moyen sur 5 ans par canal et par cohorte")
fig.update_xaxes(tickangle=-30); fig.update_layout(xaxis_title="canal", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_3_pnb5_canal_cohorte"); del df_p5c

### 9.3 — PNB 5 ans moyen par tranche_revenu et par cohorte

In [ ]:
df_p5c = (cli_5y.groupby(["cohorte", "tranche_revenu"], observed=True)["pnb_5y"].mean().round(0).reset_index())
df_p5c.head()

In [ ]:
fig = px.bar(df_p5c.astype({"cohorte": str}), x="tranche_revenu", y="pnb_5y", color="cohorte", barmode="group",
             category_orders={"tranche_revenu": ORDERS["tranche_revenu"]}, title="PNB moyen sur 5 ans par tranche_revenu et par cohorte")
fig.update_xaxes(tickangle=-30); fig.update_layout(xaxis_title="tranche_revenu", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_3_pnb5_tranche_revenu_cohorte"); del df_p5c

### 9.3 — PNB 5 ans moyen par residence et par cohorte

In [ ]:
df_p5c = (cli_5y.groupby(["cohorte", "residence"], observed=True)["pnb_5y"].mean().round(0).reset_index())
df_p5c.head()

In [ ]:
fig = px.bar(df_p5c.astype({"cohorte": str}), x="residence", y="pnb_5y", color="cohorte", barmode="group",
             category_orders={"residence": ORDERS["residence"]}, title="PNB moyen sur 5 ans par residence et par cohorte")
fig.update_xaxes(tickangle=-30); fig.update_layout(xaxis_title="residence", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_3_pnb5_residence_cohorte"); del df_p5c

### 9.3 — PNB 5 ans moyen par tranche_age et par cohorte

In [ ]:
df_p5c = (cli_5y.groupby(["cohorte", "tranche_age"], observed=True)["pnb_5y"].mean().round(0).reset_index())
df_p5c.head()

In [ ]:
fig = px.bar(df_p5c.astype({"cohorte": str}), x="tranche_age", y="pnb_5y", color="cohorte", barmode="group",
             category_orders={"tranche_age": ORDERS["tranche_age"]}, title="PNB moyen sur 5 ans par tranche_age et par cohorte")
fig.update_xaxes(tickangle=-30); fig.update_layout(xaxis_title="tranche_age", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_3_pnb5_tranche_age_cohorte"); del df_p5c

### 9.3 — PNB 5 ans moyen par csp et par cohorte

In [ ]:
df_p5c = (cli_5y.groupby(["cohorte", "csp"], observed=True)["pnb_5y"].mean().round(0).reset_index())
df_p5c.head()

In [ ]:
fig = px.bar(df_p5c.astype({"cohorte": str}), x="csp", y="pnb_5y", color="cohorte", barmode="group",
             category_orders={"csp": ORDERS["csp"]}, title="PNB moyen sur 5 ans par csp et par cohorte")
fig.update_xaxes(tickangle=-30); fig.update_layout(xaxis_title="csp", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_3_pnb5_csp_cohorte"); del df_p5c

### 9.4 Decomposition du PNB 5 ans par famille (contribution moyenne par client)

In [ ]:
rows = []
for fam in FAM_ORDER:
    v = panel[(panel["tenure"] < H5) & (panel["family"] == fam) & (panel["identifiant_tiers"].isin(elig5))]["nbi"].sum()
    rows.append({"famille": FAM_LABELS[fam], "pnb_5y_moyen": round(v / max(len(elig5), 1), 0)})
df_fam5 = pd.DataFrame(rows)
df_fam5

In [ ]:
fig = px.bar(df_fam5, x="famille", y="pnb_5y_moyen", text="pnb_5y_moyen", color="famille",
             title="PNB 5 ans moyen par client, decompose par famille", color_discrete_map=FAM_COLOR)
fig.update_traces(textposition="outside"); fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="PNB 5 ans moyen (EUR)")
save_html(fig, "9_4_pnb5_par_famille"); del df_fam5

### 9.5 PNB 5 ans par famille et par cohorte (empile)

In [ ]:
rows = []
for coh in sorted(cli_5y["cohorte"].dropna().unique()):
    ids = set(cli_5y.index[cli_5y["cohorte"] == coh])
    for fam in FAM_ORDER:
        v = panel[(panel["tenure"] < H5) & (panel["family"] == fam) & (panel["identifiant_tiers"].isin(ids))]["nbi"].sum()
        rows.append({"cohorte": int(coh), "famille": FAM_LABELS[fam], "pnb_5y_moyen": round(v / max(len(ids), 1), 0)})
df_fam5c = pd.DataFrame(rows)
df_fam5c.pivot(index="cohorte", columns="famille", values="pnb_5y_moyen")

In [ ]:
fig = px.bar(df_fam5c.astype({"cohorte": str}), x="cohorte", y="pnb_5y_moyen", color="famille",
             title="PNB 5 ans moyen par famille et par cohorte", color_discrete_map=FAM_COLOR)
fig.update_layout(xaxis_title="Cohorte", yaxis_title="PNB 5 ans moyen (EUR)", legend_title="Famille")
save_html(fig, "9_5_pnb5_famille_cohorte"); del df_fam5c

### 9.6 PNB moyen par annee de vie (Y1 -> Y5) et par cohorte

Pour chaque cohorte a 5 ans de recul : combien un client genere en moyenne sa 1re annee,
sa 2e, ... sa 5e. Base FLUX (NBI total) ; un client qui a churne compte 0 sur ses annees
post-depart, donc Y1+...+Y5 = le PNB total 5 ans.

In [ ]:
flux5 = flux[(flux["tenure"] < H5) & (flux["identifiant_tiers"].isin(elig5))]
per = flux5.assign(annee_vie=flux5["tenure"] // 12 + 1).groupby(["identifiant_tiers", "annee_vie"])["nbi"].sum()
grid = pd.MultiIndex.from_product([cli_5y.index, [1, 2, 3, 4, 5]], names=["identifiant_tiers", "annee_vie"])
per = per.reindex(grid, fill_value=0).reset_index()
per["cohorte"] = per["identifiant_tiers"].map(cohort_of)
df_year = per.groupby(["cohorte", "annee_vie"], as_index=False)["nbi"].mean().round(0)
df_year["annee_vie"] = "Y" + df_year["annee_vie"].astype(int).astype(str)
del flux5, per; df_year.pivot(index="cohorte", columns="annee_vie", values="nbi")

In [ ]:
fig = px.bar(df_year.astype({"cohorte": str}), x="annee_vie", y="nbi", color="cohorte", barmode="group",
             text="nbi", title="PNB moyen par annee de vie et par cohorte")
fig.update_layout(xaxis_title="Annee de vie", yaxis_title="PNB moyen (EUR)", legend_title="Cohorte")
save_html(fig, "9_6_pnb_par_annee_vie"); del df_year

### 9.7 Tableau — PNB total moyen sur 5 ans par cohorte

In [ ]:
df_total_5y = (cli_5y.groupby("cohorte")
               .agg(n_clients=("pnb_5y", "size"), pnb_5ans_moyen=("pnb_5y", "mean"), pnb_5ans_median=("pnb_5y", "median"))
               .round(0).astype("Int64").reset_index())
print("PNB total moyen sur 5 ans par cohorte :")
print(df_total_5y.to_string(index=False))
print(f"\nEnsemble : {cli_5y['pnb_5y'].mean():,.0f} EUR moyen / client sur 5 ans (n={len(cli_5y):,})")
df_total_5y